In [ ]:
# Library imports
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
sys.path.append(os.path.abspath('..'))

# File imports
from env import InventoryEnv
from miscelaneous import read_instance
from policies.perfect_hindsight_policy import PerfectHindsightPolicy
from policies.linear_programming_exact import LinearProgrammingExactPolicy
from policies.auxiliaries.imitation_learning_model import ImitationLearningNet

In [3]:
def run_episode(env, policy, instance_file, output_dir):
    # policy is any expert with the BasePolicy interface (reset(instance)/act(state)) -
    # PerfectHindsightPolicy precomputes its whole action sequence in reset() and
    # replays it, while LinearProgrammingExactPolicy (LPE) decides live at each step, but
    # both are driven identically here since neither needs special-casing.

    num_warehouses = env.num_warehouses
    num_customers = env.num_customers
    capacity_distribution = env.capacity_distribution

    instance = read_instance(instance_file)
    env.set_instance(instance)
    policy.reset(instance)  # points env.get_state at the identity function - both experts need the raw state dict
    state, info = env.reset()

    done = False
    data = []

    while not done:

        action = policy.act(state)

        warehouses_distance = state["warehouses_distance"]
        warehouses_capacity = state["warehouses_capacity"]
        
        row = []

        for i in range(state['static_info']['num_warehouses']):
            row.append(warehouses_distance[i] / 212.13)
            row.append(warehouses_capacity[i] / state['static_info']['warehouses_initial_capacity'][i])

        row.append(action)

        data.append(row)

        state, reward, done, truncated, _ = env.step(action)

    #Save data in a csv file
    df = pd.DataFrame(data, columns=[f"warehouses_distance_{i}" for i in range(num_warehouses)] + [f"warehouses_capacity_{i}" for i in range(num_warehouses)] + ["action"])

    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path / f"data_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv", mode='a', header=False, index=False)

In [4]:
# Which expert policy the neural net will imitate - "perfect_hindsight" (the original
# oracle-with-future-knowledge expert) or "linear_programming_exact" (LPE, the same
# online, no-hindsight expert DecisionTreePolicy imitates - see
# train_decision_tree.ipynb). Each writes to its own output folder so the two datasets
# never mix; nn_model() below still reads from perfect_agent_data by default, so point
# it at linear_programming_exact_agent_data too if training on the LPE-imitation data.
EXPERT_OUTPUT_DIRS = {
    "perfect_hindsight": "imitation_learning_training/perfect_agent_data",
    "linear_programming_exact": "imitation_learning_training/linear_programming_exact_agent_data",
}


def make_expert_policy(expert_name, env, num_warehouses):
    if expert_name == "perfect_hindsight":
        return PerfectHindsightPolicy(env)
    elif expert_name == "linear_programming_exact":
        return LinearProgrammingExactPolicy(env, num_warehouses)
    else:
        raise ValueError(f"Unknown expert_name: {expert_name!r} (expected 'perfect_hindsight' or 'linear_programming_exact')")


def export_data(num_warehouses, num_customers, capacity_distribution, expert_name="perfect_hindsight"):

    env = InventoryEnv(num_warehouses, num_customers, capacity_distribution)
    policy = make_expert_policy(expert_name, env, num_warehouses)

    num_instances = 1000
    instances = [f'../instances/instances_train_imitation_learning/instances_seed_{i}.json' for i in range(51,num_instances+51)]

    output_dir = EXPERT_OUTPUT_DIRS[expert_name]

    for instance in instances:
        run_episode(env, policy, instance, output_dir)

In [ ]:
EXPERT_NAME = "perfect_hindsight"  # or "linear_programming_exact"

for num_warehouses in [2, 3, 4, 5]:
    for num_customers in [50, 100, 200, 400]:
        for capacity_distribution in ["uniform", "uneven"]:
            export_data(num_warehouses, num_customers, capacity_distribution, expert_name=EXPERT_NAME)

Set parameter Username
Set parameter LicenseID to value 2776932
Academic license - for non-commercial use only - expires 2027-02-09


In [ ]:
def nn_model(num_warehouses, num_customers, capacity_distribution, input_dir="imitation_learning_training/perfect_agent_data"):

    # Read the data from the CSV file
    df = pd.read_csv(
        #f"imitation_learning_training/linear_programming_exact_agent_data/data_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv",
        f"{input_dir}/data_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.csv",
        header=None
    )
    # Rename columns to alternate distance/capacity
    columns = []
    for i in range(num_warehouses):
        columns.append(f"d_{i}")
        columns.append(f"c_{i}")
    columns.append("action")
    df.columns = columns

    # Prepare the input and output data
    X = df.drop(columns=["action"]).values.astype(np.float32)
    y = df["action"].values.astype(np.int64)

    # Split the data into training and testing sets (same split as before)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    # Further split the training data into train/validation - mirrors Keras' fit(..., validation_split=0.2)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

    X_train_t, y_train_t = torch.tensor(X_train), torch.tensor(y_train)
    X_val_t, y_val_t = torch.tensor(X_val), torch.tensor(y_val)
    X_test_t, y_test_t = torch.tensor(X_test), torch.tensor(y_test)

    # Build the neural network model - same architecture as the Keras version
    # (Dense 64 -> Dense 16 -> Dense num_warehouses, ReLU/ReLU), defined once in
    # policies/imitation_learning_model.py and shared with ImitationLearningPolicy so
    # training and inference can never drift apart
    model = ImitationLearningNet(num_warehouses)
    optimizer = torch.optim.Adam(model.parameters())
    # sparse_categorical_crossentropy on integer labels == CrossEntropyLoss on raw logits
    criterion = nn.CrossEntropyLoss()

    train_loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True
    )

    # Train the model
    epochs = 25
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_loss = criterion(val_logits, y_val_t).item()
            val_accuracy = (val_logits.argmax(dim=1) == y_val_t).float().mean().item()
        print(f"Epoch {epoch+1}/{epochs} - val_loss: {val_loss:.4f} - val_accuracy: {val_accuracy:.4f}")

    # Evaluate the model
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        accuracy = (test_logits.argmax(dim=1) == y_test_t).float().mean().item()
    print(f"Accuracy: {accuracy}")

    # Save the model
    torch.save(
        model.state_dict(),
        f"imitation_learning_training/imitation_learning_models/model_nn_w_{num_warehouses}_c_{num_customers}_d_{capacity_distribution}.pt"
    )

In [ ]:
num_warehouses_options = [2, 3, 4, 5]
num_customers_options = [50, 100, 200, 400]
capacity_distribution_options = ["uniform", "uneven"]

for num_warehouses in num_warehouses_options:
    for num_customers in num_customers_options:
        for capacity_distribution in capacity_distribution_options:
            nn_model(num_warehouses, num_customers, capacity_distribution, input_dir=EXPERT_OUTPUT_DIRS[EXPERT_NAME])

Epoch 1/25 - val_loss: 0.1324 - val_accuracy: 0.9413
Epoch 2/25 - val_loss: 0.1282 - val_accuracy: 0.9417
Epoch 3/25 - val_loss: 0.1282 - val_accuracy: 0.9411
Epoch 4/25 - val_loss: 0.1278 - val_accuracy: 0.9434
Epoch 5/25 - val_loss: 0.1280 - val_accuracy: 0.9417
Epoch 6/25 - val_loss: 0.1284 - val_accuracy: 0.9416
Epoch 7/25 - val_loss: 0.1286 - val_accuracy: 0.9427
Epoch 8/25 - val_loss: 0.1277 - val_accuracy: 0.9434
Epoch 9/25 - val_loss: 0.1292 - val_accuracy: 0.9425
Epoch 10/25 - val_loss: 0.1274 - val_accuracy: 0.9434
Epoch 11/25 - val_loss: 0.1291 - val_accuracy: 0.9426
Epoch 12/25 - val_loss: 0.1274 - val_accuracy: 0.9411
Epoch 13/25 - val_loss: 0.1274 - val_accuracy: 0.9424
Epoch 14/25 - val_loss: 0.1272 - val_accuracy: 0.9426
Epoch 15/25 - val_loss: 0.1295 - val_accuracy: 0.9425
Epoch 16/25 - val_loss: 0.1270 - val_accuracy: 0.9430
Epoch 17/25 - val_loss: 0.1271 - val_accuracy: 0.9438
Epoch 18/25 - val_loss: 0.1335 - val_accuracy: 0.9400
Epoch 19/25 - val_loss: 0.1279 - val_